<a href="https://colab.research.google.com/github/maheshvlfm-coder/Ecom_delivery_optimisation/blob/main/Delivery_optimisation_for_E_Commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instructions
Step 1: Download sample files to understand the format

Step 2: Upload your warehouse, stock, and orders data (CSV or Excel)

Step 3: Process and optimize delivery routes with mapping

Step 4: Download optimized routes and comprehensive analysis

Step 5: View interactive maps with clusters and routes**

In [ ]:
# -*- coding: utf-8 -*-
""" Delivery Optimization System - Enhanced Version with Robust Coordinate Validation
Enhanced version with improved coordinate validation, mapping, clustering, and route optimization
Compatible with Google Colab and Jupyter Notebook
Features: Robust missing data handling, interactive maps, clustering, optimized routes
"""

# Install required packages
!pip install folium googlemaps requests pandas numpy ipywidgets geopy openpyxl scikit-learn plotly

import pandas as pd
import numpy as np
import folium
from folium import plugins
import googlemaps
import requests
import json
import io
import base64
import hashlib
import zipfile
import os
import warnings
from datetime import datetime
from typing import Callable, List, Dict, Tuple, Optional
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files
import geopy.distance
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

class AmazonDeliveryOptimizer:
    def __init__(self):
        self.warehouse_data = None
        self.stock_data = None
        self.orders_data = None
        self.delivery_plan = None
        self.missing_data_records = None
        self.route_optimization = None
        self.cluster_data = None
        self.api_key = "YOUR_GOOGLE_MAPS_API_KEY"  # Replace with your actual API key
        self.gmaps = None
        self.setup_ui()

    def setup_ui(self):
        """Setup the user interface with interactive buttons"""
        # Title and description
        title = widgets.HTML(value="""
        <div style='text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    color: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
            <h1 style='margin: 0; font-size: 28px;'>🚚 Smart Route Planning & Warehouse Optimization</h1>
            <p style='margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;'>
                Advanced delivery optimization with robust coordinate validation and mapping
            </p>
        </div>
        """)

        instructions = widgets.HTML(value="""
        <div style='background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #007bff; margin-bottom: 20px;'>
            <h3 style='color: #007bff; margin-top: 0;'>📋 Instructions</h3>
            <p><strong>Step 1:</strong> Download sample files to understand the format</p>
            <p><strong>Step 2:</strong> Upload your warehouse, stock, and orders data (CSV or Excel)</p>
            <p><strong>Step 3:</strong> Process and optimize delivery routes with mapping</p>
            <p><strong>Step 4:</strong> Download optimized routes and comprehensive analysis</p>
            <p><strong>Step 5:</strong> View interactive maps with clusters and routes</p>
        </div>
        """)

        # Create buttons with improved styling
        button_style = {'description_width': 'initial', 'button_color': '#007bff'}

        self.download_samples_btn = widgets.Button(
            description='📥 Download Sample Files',
            style=button_style,
            layout=widgets.Layout(width='200px', height='40px')
        )

        self.upload_btn = widgets.Button(
            description='📤 Upload Data Files',
            style=button_style,
            layout=widgets.Layout(width='200px', height='40px')
        )

        self.process_btn = widgets.Button(
            description='⚡ Process & Map',
            style=button_style,
            layout=widgets.Layout(width='200px', height='40px')
        )

        self.download_routes_btn = widgets.Button(
            description='🗺️ Download Routes',
            style={'description_width': 'initial', 'button_color': '#28a745'},
            layout=widgets.Layout(width='200px', height='40px')
        )

        self.download_btn = widgets.Button(
            description='💾 Download Complete Analysis',
            style={'description_width': 'initial', 'button_color': '#dc3545'},
            layout=widgets.Layout(width='220px', height='40px')
        )

        # Status display
        self.status_output = widgets.Output()

        # Button event handlers
        self.download_samples_btn.on_click(self.download_sample_files)
        self.upload_btn.on_click(self.upload_files)
        self.process_btn.on_click(self.process_data)
        self.download_routes_btn.on_click(self.download_route_optimization)
        self.download_btn.on_click(self.download_complete_analysis)

        # Layout
        button_row1 = widgets.HBox([
            self.download_samples_btn,
            self.upload_btn,
            self.process_btn
        ], layout=widgets.Layout(justify_content='space-around', margin='10px 0'))

        button_row2 = widgets.HBox([
            self.download_routes_btn,
            self.download_btn
        ], layout=widgets.Layout(justify_content='space-around', margin='10px 0'))

        self.ui = widgets.VBox([
            title,
            instructions,
            button_row1,
            button_row2,
            self.status_output
        ])

        display(self.ui)

    def is_valid_coordinate(self, value):
        """Check if a single coordinate value is valid"""
        try:
            # Convert to float if not already
            if isinstance(value, str):
                if value.strip() == '' or value.strip().lower() in ['na', 'nan', 'null', 'none']:
                    return False
                value = float(value)

            # Check if value is a number
            if not isinstance(value, (int, float)):
                return False

            # Check for NaN, infinity, or zero
            if pd.isna(value) or math.isnan(value) or math.isinf(value):
                return False

            # Check if coordinate is exactly zero (often invalid)
            if value == 0.0 or value == 0:
                return False

            # Check if it's a finite number
            if not math.isfinite(value):
                return False

            return True

        except (ValueError, TypeError, OverflowError):
            return False

    def validate_coordinates(self, df, lat_col, lon_col):
        """Validate and separate records with missing coordinates"""
        valid_records = []
        missing_records = []

        print(f"   🔍 Validating {len(df)} records for coordinate issues...")

        for idx, row in df.iterrows():
            lat = row.get(lat_col)
            lon = row.get(lon_col)

            # Check if both coordinates are valid
            lat_valid = self.is_valid_coordinate(lat)
            lon_valid = self.is_valid_coordinate(lon)

            if lat_valid and lon_valid:
                # Additional range check for latitude and longitude
                try:
                    lat_float = float(lat)
                    lon_float = float(lon)

                    # Check coordinate ranges (valid Earth coordinates)
                    if (-90 <= lat_float <= 90) and (-180 <= lon_float <= 180):
                        # Convert to ensure proper float type
                        row_dict = row.to_dict()
                        row_dict[lat_col] = lat_float
                        row_dict[lon_col] = lon_float
                        valid_records.append(row_dict)
                    else:
                        # Out of range coordinates
                        row_dict = row.to_dict()
                        row_dict['Missing_Data_Remark'] = f'Coordinates out of range: lat={lat}, lon={lon}'
                        missing_records.append(row_dict)

                except (ValueError, TypeError):
                    # Conversion failed
                    row_dict = row.to_dict()
                    row_dict['Missing_Data_Remark'] = f'Invalid coordinate format: lat={lat}, lon={lon}'
                    missing_records.append(row_dict)
            else:
                # Invalid coordinates
                row_dict = row.to_dict()
                issues = []
                if not lat_valid:
                    issues.append(f'invalid lat={lat}')
                if not lon_valid:
                    issues.append(f'invalid lon={lon}')
                row_dict['Missing_Data_Remark'] = f'Invalid coordinates: {", ".join(issues)}'
                missing_records.append(row_dict)

        valid_df = pd.DataFrame(valid_records) if valid_records else pd.DataFrame()
        missing_df = pd.DataFrame(missing_records) if missing_records else pd.DataFrame()

        print(f"   ✅ Valid records: {len(valid_df)}")
        print(f"   ⚠️ Invalid records: {len(missing_df)}")

        return valid_df, missing_df

    def create_sample_files(self):
        """Create sample files with exact headers and structure from the provided data"""

        # Sample Warehouse Location Data
        warehouse_data = {
            'Number': [1, 2, 3, 4, 5],
            'FC Code': ['SVGA', 'SDEA', 'SDED', 'SDEE', 'SDEF'],
            'state': ['Andhra Pradesh', 'Delhi', 'Delhi', 'Delhi', 'Delhi'],
            'Fulfillment Center Address': [
                'CLOUDTAIL INDIA PVT LTD Amazon seller services private limited, C/o Yusen Logistics India Pvt Ltd Sy. No: 36/6 & 36/4B',
                'CLOUDTAIL INDIA PVT LTD C/o AMAZON SELLER SERVICE PVT LTD A-43, Ground floor, Mohan Cooperative Industrial Estate Mathura Road, Badarpur - 110044',
                'CLOUDTAIL INDIA PVT LTD C/o AMAZON SELLER SERVICE PVT LTD J-4/B-1 Mohan cooperative Industrial Estate Mathura Road, Delhi - 110044',
                'CLOUDTAIL INDIA PVT LTD C/o AMAZON SELLER SERVICE PVT LTD Plot No - A-28 Mohan cooperative Industrial Estate Mathura Road, Delhi - 110044',
                'CLOUDTAIL INDIA PVT LTD C/o AMAZON SELLER SERVICE PVT LTD A-33, Mohan cooperative Industrial Estate Mathura Road, Delhi - 110044'
            ],
            'latitude': [12.962231, 28.499811, 28.518124, 28.505966, 28.521863],
            'longitude': [77.599535, 77.302293, 77.293759, 77.291748, 77.291172]
        }

        # Sample Stock Data - Creating a structure with PRODUCT CODE and warehouse codes
        warehouse_codes = ['SVGA', 'SDEA', 'SDED', 'SDEE', 'SDEF', 'UDL2', 'UDL4', 'UDL5', 'PIDE', 'ZNLE',
                          'AMD1', 'SAMD', 'XWAE', 'QWVJ', 'UDL1', 'DEL2', 'DEL4', 'DEL3', 'SDEG', 'XNAB',
                          'SDEB', 'ZNJU', 'DEL5', 'SDEK', 'BLR5', 'SBLB', 'SBLC', 'XSAB', 'UBL1', 'UBL2',
                          'UBL3', 'UBL4', 'BLR7', 'QSN3', 'XSAJ', 'SBLD', 'QSGZ', 'SIDA', 'SBHA', 'BOM3',
                          'BOM4', 'BOM1', 'SPNA', 'SPNC', 'SBOA', 'QWMI', 'SBOE', 'SBOC', 'NAG1', 'XWAA',
                          'SBOB', 'UB01', 'UB02', 'UB03', 'QWVK', 'QWQI', 'QWUR', 'XWAL', 'QWUP', 'SBOG',
                          'SATA', 'SATB', 'ZNCM', 'QNPZ', 'SJAB', 'QWUQ', 'QSFM', 'MAA4', 'MAA5', 'SCJA',
                          'SCJB', 'SMAB', 'QSN1', 'HYD8', 'SHYB', 'XSAD', 'UHY1', 'UHY2', 'UHY3', 'SHYC',
                          'UDL3', 'SDEH', 'SLKA', 'SLKB', 'ZNJT', 'SCCA', 'SCCB', 'XEAA', 'SCCC', 'QEA3',
                          'SCCF', 'SGAA', 'DEX3', 'PNQ2', 'DEX8', 'AMD2', 'DEL8_DED5', 'DED3', 'DED4', 'BLR4',
                          'BLR8', 'FBHB', 'FIDA', 'PNQ3', 'BOM7', 'BOM5', 'ISK3', 'ATX1', 'JPX2', 'JPX1',
                          'CJB1', 'HYD8_HYD3', 'LKO1', 'SLDK', 'CCX1', 'CCX2']

        stock_data = {
            'PRODUCT CODE': [83227464, 39433826, 33705422, 20527814, 26924703]
        }

        # Add stock quantities for each warehouse (sample data)
        np.random.seed(42)  # For reproducible sample data
        for warehouse in warehouse_codes:
            stock_data[warehouse] = np.random.randint(1, 101, 5).tolist()

        # Sample Orders Data
        orders_data = {
            'Number': [1, 2, 3, 4, 5],
            'Address': ['226 Nehru Nagar', '33 Park Street', '50 MG Road', '34 Park Street', '207 MG Road'],
            'City': ['Mysore', 'Kanpur', 'Bareilly', 'Hyderabad', 'Indore'],
            'State': ['Karnataka', 'Uttar Pradesh', 'Uttar Pradesh', 'Telangana', 'Madhya Pradesh'],
            'Postal Code': [570001, 208001, 243001, 500001, 452001],
            'Latitude': [12.289645, 26.439903, 28.374841, 17.376850, 22.699305],
            'Longitude': [76.660456, 80.348642, 79.432515, 78.464528, 75.863378],
            'PRODUCT CODE': [83227464, 39433826, 33705422, 20527814, 26924703],
            'ORDER_QTY': [36, 9, 62, 19, 52]
        }

        # Create DataFrames
        warehouse_df = pd.DataFrame(warehouse_data)
        stock_df = pd.DataFrame(stock_data)
        orders_df = pd.DataFrame(orders_data)

        return warehouse_df, stock_df, orders_df

    def create_clusters_and_map(self, warehouse_df, orders_df):
        """Create clusters and interactive map visualization"""
        print("🗺️ Creating clusters and interactive map...")

        try:
            # Ensure all coordinates are valid before proceeding
            all_coords = []
            labels = []
            types = []

            # Add warehouse coordinates with validation
            for _, wh in warehouse_df.iterrows():
                lat = float(wh['latitude'])
                lon = float(wh['longitude'])

                # Double-check coordinates are valid
                if self.is_valid_coordinate(lat) and self.is_valid_coordinate(lon):
                    all_coords.append([lat, lon])
                    labels.append(f"WH: {wh['FC Code']}")
                    types.append('warehouse')

            # Add order coordinates with validation
            for _, order in orders_df.iterrows():
                lat = float(order['Latitude'])
                lon = float(order['Longitude'])

                # Double-check coordinates are valid
                if self.is_valid_coordinate(lat) and self.is_valid_coordinate(lon):
                    all_coords.append([lat, lon])
                    labels.append(f"Order: {order['Number']} - {order['City']}")
                    types.append('order')

            if len(all_coords) < 2:
                print("⚠️ Not enough valid coordinates for clustering")
                return None, None

            # Perform clustering
            n_clusters = min(8, len(all_coords))  # Maximum 8 clusters
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(all_coords)

            # Create folium map
            center_lat = np.mean([coord[0] for coord in all_coords])
            center_lon = np.mean([coord[1] for coord in all_coords])

            # Validate center coordinates
            if not (self.is_valid_coordinate(center_lat) and self.is_valid_coordinate(center_lon)):
                center_lat, center_lon = 20.5937, 78.9629  # Default to India center

            m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

            # Color palette for clusters
            colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'darkblue']

            # Add markers for warehouses
            wh_count = 0
            for _, wh in warehouse_df.iterrows():
                lat = float(wh['latitude'])
                lon = float(wh['longitude'])

                if self.is_valid_coordinate(lat) and self.is_valid_coordinate(lon):
                    color = colors[clusters[wh_count] % len(colors)]
                    folium.Marker(
                        location=[lat, lon],
                        popup=f"""
                        <b>Warehouse: {wh['FC Code']}</b><br>
                        State: {wh['state']}<br>
                        Cluster: {clusters[wh_count]}<br>
                        Address: {str(wh['Fulfillment Center Address'])[:100]}...
                        """,
                        icon=folium.Icon(color=color, icon='home', prefix='fa'),
                        tooltip=f"Warehouse: {wh['FC Code']}"
                    ).add_to(m)
                    wh_count += 1

            # Add markers for orders
            order_count = 0
            for _, order in orders_df.iterrows():
                lat = float(order['Latitude'])
                lon = float(order['Longitude'])

                if self.is_valid_coordinate(lat) and self.is_valid_coordinate(lon):
                    color = colors[clusters[wh_count + order_count] % len(colors)]
                    folium.CircleMarker(
                        location=[lat, lon],
                        popup=f"""
                        <b>Order: {order['Number']}</b><br>
                        City: {order['City']}, {order['State']}<br>
                        Product: {order['PRODUCT CODE']}<br>
                        Quantity: {order['ORDER_QTY']}<br>
                        Cluster: {clusters[wh_count + order_count]}
                        """,
                        radius=8,
                        color=color,
                        fillColor=color,
                        fillOpacity=0.7,
                        tooltip=f"Order: {order['Number']} - {order['City']}"
                    ).add_to(m)
                    order_count += 1

            # Add cluster centers
            for i, center in enumerate(kmeans.cluster_centers_):
                if self.is_valid_coordinate(center[0]) and self.is_valid_coordinate(center[1]):
                    folium.Marker(
                        location=[center[0], center[1]],
                        popup=f"Cluster {i} Center",
                        icon=folium.Icon(color='black', icon='star', prefix='fa')
                    ).add_to(m)

            # Save map
            m.save('delivery_optimization_map.html')

            # Store cluster data
            cluster_data = {
                'coordinates': all_coords,
                'labels': labels,
                'types': types,
                'clusters': clusters.tolist(),
                'n_clusters': n_clusters
            }

            print(f"✅ Map created with {len(all_coords)} valid locations and {n_clusters} clusters")
            return cluster_data, m

        except Exception as e:
            print(f"❌ Error creating map: {str(e)}")
            return None, None

    def safe_distance_calculation(self, coord1, coord2):
        """Safely calculate distance between two coordinates"""
        try:
            lat1, lon1 = coord1
            lat2, lon2 = coord2

            # Validate all coordinates
            if not all([
                self.is_valid_coordinate(lat1),
                self.is_valid_coordinate(lon1),
                self.is_valid_coordinate(lat2),
                self.is_valid_coordinate(lon2)
            ]):
                return float('inf')  # Return infinite distance for invalid coords

            # Convert to float to ensure proper type
            lat1, lon1, lat2, lon2 = float(lat1), float(lon1), float(lat2), float(lon2)

            # Calculate distance
            distance = geopy.distance.distance((lat1, lon1), (lat2, lon2)).kilometers

            # Ensure result is valid
            if not math.isfinite(distance) or distance < 0:
                return float('inf')

            return distance

        except Exception as e:
            print(f"   ⚠️ Distance calculation error: {str(e)}")
            return float('inf')

    def optimize_delivery_routes(self, warehouse_df, orders_df):
        """Create optimized delivery plan with valid data"""

        print("🔄 Analyzing warehouse-order distances...")

        # Create delivery assignments based on proximity
        delivery_assignments = []
        skipped_orders = 0

        for _, order in orders_df.iterrows():
            order_lat = float(order['Latitude'])
            order_lon = float(order['Longitude'])

            # Validate order coordinates
            if not (self.is_valid_coordinate(order_lat) and self.is_valid_coordinate(order_lon)):
                skipped_orders += 1
                continue

            # Find closest warehouse
            min_distance = float('inf')
            best_warehouse = None

            for _, warehouse in warehouse_df.iterrows():
                wh_lat = float(warehouse['latitude'])
                wh_lon = float(warehouse['longitude'])

                # Validate warehouse coordinates
                if not (self.is_valid_coordinate(wh_lat) and self.is_valid_coordinate(wh_lon)):
                    continue

                # Calculate distance safely
                distance = self.safe_distance_calculation(
                    (order_lat, order_lon),
                    (wh_lat, wh_lon)
                )

                if distance < min_distance and distance != float('inf'):
                    min_distance = distance
                    best_warehouse = warehouse

            # Only create assignment if we found a valid warehouse
            if best_warehouse is not None and min_distance != float('inf'):
                assignment = {
                    'Order_ID': order['Number'],
                    'Product_Code': order['PRODUCT CODE'],
                    'Order_Qty': order['ORDER_QTY'],
                    'Customer_Address': order['Address'],
                    'Customer_City': order['City'],
                    'Customer_State': order['State'],
                    'Customer_Lat': order_lat,
                    'Customer_Lon': order_lon,
                    'Assigned_Warehouse': best_warehouse['FC Code'],
                    'Warehouse_State': best_warehouse['state'],
                    'Warehouse_Lat': float(best_warehouse['latitude']),
                    'Warehouse_Lon': float(best_warehouse['longitude']),
                    'Distance_KM': round(min_distance, 2),
                    'Estimated_Delivery_Time': self.calculate_delivery_time(min_distance),
                    'Delivery_Cost': round(min_distance * 2.5, 2)  # ₹2.5 per km
                }
                delivery_assignments.append(assignment)
            else:
                skipped_orders += 1

        if skipped_orders > 0:
            print(f"   ⚠️ Skipped {skipped_orders} orders due to coordinate issues")

        print("🔄 Generating route optimization...")

        # Create delivery plan DataFrame
        delivery_df = pd.DataFrame(delivery_assignments)

        if delivery_df.empty:
            print("❌ No valid delivery assignments could be created!")
            return None

        # Add summary statistics
        summary = {
            'Total_Orders': len(delivery_df),
            'Total_Distance_KM': delivery_df['Distance_KM'].sum(),
            'Total_Cost': delivery_df['Delivery_Cost'].sum(),
            'Average_Distance': delivery_df['Distance_KM'].mean(),
            'Warehouses_Used': delivery_df['Assigned_Warehouse'].nunique(),
            'Skipped_Orders': skipped_orders,
            'Missing_Records': len(self.missing_data_records) if self.missing_data_records is not None else 0
        }

        return {
            'delivery_assignments': delivery_df,
            'summary': summary,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

    def optimize_routes_by_warehouse(self, delivery_df):
        """Create optimized routes for each warehouse"""
        print("🚛 Optimizing delivery routes by warehouse...")

        route_optimization = {}

        for warehouse in delivery_df['Assigned_Warehouse'].unique():
            wh_orders = delivery_df[delivery_df['Assigned_Warehouse'] == warehouse].copy()

            if len(wh_orders) == 0:
                continue

            # Sort orders by distance for basic optimization
            wh_orders = wh_orders.sort_values('Distance_KM')

            # Calculate cumulative distance and time for route
            total_distance = wh_orders['Distance_KM'].sum()
            total_orders = len(wh_orders)

            # Estimate delivery sequence and times
            cumulative_distance = 0
            route_details = []

            for idx, (_, order) in enumerate(wh_orders.iterrows()):
                cumulative_distance += order['Distance_KM']

                route_detail = {
                    'Sequence': idx + 1,
                    'Order_ID': order['Order_ID'],
                    'Customer_City': order['Customer_City'],
                    'Customer_State': order['Customer_State'],
                    'Product_Code': order['Product_Code'],
                    'Order_Qty': order['Order_Qty'],
                    'Distance_KM': order['Distance_KM'],
                    'Cumulative_Distance': round(cumulative_distance, 2),
                    'Estimated_Time': self.calculate_delivery_time(order['Distance_KM']),
                    'Delivery_Cost': order['Delivery_Cost'],
                    'Customer_Coordinates': f"{order['Customer_Lat']}, {order['Customer_Lon']}"
                }
                route_details.append(route_detail)

            # Calculate route statistics
            route_stats = {
                'Warehouse_Code': warehouse,
                'Total_Orders': total_orders,
                'Total_Distance_KM': round(total_distance, 2),
                'Total_Cost': round(wh_orders['Delivery_Cost'].sum(), 2),
                'Average_Distance': round(total_distance / total_orders, 2),
                'Estimated_Total_Time': self.calculate_route_time(total_distance),
                'Route_Efficiency_Score': self.calculate_efficiency_score(wh_orders)
            }

            route_optimization[warehouse] = {
                'route_details': pd.DataFrame(route_details),
                'route_stats': route_stats,
                'warehouse_coordinates': (
                    wh_orders.iloc[0]['Warehouse_Lat'],
                    wh_orders.iloc[0]['Warehouse_Lon']
                )
            }

        return route_optimization

    def calculate_delivery_time(self, distance_km):
        """Calculate estimated delivery time based on distance"""
        if distance_km <= 50:
            return "Same Day"
        elif distance_km <= 200:
            return "Next Day"
        elif distance_km <= 500:
            return "2-3 Days"
        else:
            return "3-5 Days"

    def calculate_route_time(self, total_distance):
        """Calculate estimated total route time"""
        # Assuming average speed of 40 km/h with stops
        driving_time = total_distance / 40
        # Add time for stops (15 minutes per stop)
        stop_time = 0.25  # 15 minutes in hours
        return f"{driving_time + stop_time:.1f} hours"

    def calculate_efficiency_score(self, orders_df):
        """Calculate route efficiency score (0-100)"""
        # Simple efficiency based on distance variance
        distances = orders_df['Distance_KM']
        avg_distance = distances.mean()
        variance = distances.var()

        # Lower variance = higher efficiency
        efficiency = max(0, 100 - (variance / avg_distance * 10))
        return round(efficiency, 1)

    def download_sample_files(self, btn):
        """Generate and download sample files"""
        with self.status_output:
            clear_output(wait=True)
            print("🔄 Creating sample files...")

            try:
                # Create sample data
                warehouse_df, stock_df, orders_df = self.create_sample_files()

                # Create Excel files
                warehouse_df.to_excel('sample_warehouse_location.xlsx', index=False, sheet_name='amazon_warehouses')
                stock_df.to_excel('sample_stock.xlsx', index=False, sheet_name='Sheet1')
                orders_df.to_excel('sample_orders.xlsx', index=False, sheet_name='orders')

                print("✅ Sample files created successfully!")
                print("📁 Files created:")
                print("   • sample_warehouse_location.xlsx")
                print("   • sample_stock.xlsx")
                print("   • sample_orders.xlsx")
                print("\n🔽 Click below to download each file:")

                # Download files
                files.download('sample_warehouse_location.xlsx')
                files.download('sample_stock.xlsx')
                files.download('sample_orders.xlsx')

                print("\n💡 Use these sample files as templates for your data!")

            except Exception as e:
                print(f"❌ Error creating sample files: {str(e)}")

    def upload_files(self, btn):
        """Handle file upload"""
        with self.status_output:
            clear_output(wait=True)
            print("📤 Please upload your data files...")
            print("Expected files:")
            print("  • Warehouse location file (Excel/CSV)")
            print("  • Stock data file (Excel/CSV)")
            print("  • Orders data file (Excel/CSV)")

            uploaded = files.upload()

            if uploaded:
                print(f"\n✅ {len(uploaded)} file(s) uploaded successfully!")
                for filename in uploaded.keys():
                    print(f"   • {filename}")
                print("\n🔄 Processing uploaded files...")

                try:
                    # Process uploaded files
                    self.load_data_files(list(uploaded.keys()))
                    print("✅ Files processed successfully!")
                    print("📊 Data loaded and ready for optimization")

                except Exception as e:
                    print(f"❌ Error processing files: {str(e)}")
                    print("Please check your file formats and try again.")
            else:
                print("❌ No files uploaded.")

    def load_data_files(self, filenames):
        """Load and validate uploaded data files"""

        for filename in filenames:
            try:
                # Determine file type and load accordingly
                if filename.endswith('.xlsx') or filename.endswith('.xls'):
                    if 'warehouse' in filename.lower() or 'location' in filename.lower():
                        self.warehouse_data = pd.read_excel(filename)
                        print(f"   📍 Warehouse data loaded: {self.warehouse_data.shape}")

                    elif 'stock' in filename.lower():
                        self.stock_data = pd.read_excel(filename)
                        print(f"   📦 Stock data loaded: {self.stock_data.shape}")

                    elif 'order' in filename.lower():
                        self.orders_data = pd.read_excel(filename)
                        print(f"   🛒 Orders data loaded: {self.orders_data.shape}")

                elif filename.endswith('.csv'):
                    df = pd.read_csv(filename)
                    if 'warehouse' in filename.lower() or 'location' in filename.lower():
                        self.warehouse_data = df
                        print(f"   📍 Warehouse data loaded: {self.warehouse_data.shape}")

                    elif 'stock' in filename.lower():
                        self.stock_data = df
                        print(f"   📦 Stock data loaded: {self.stock_data.shape}")

                    elif 'order' in filename.lower():
                        self.orders_data = df
                        print(f"   🛒 Orders data loaded: {self.orders_data.shape}")

            except Exception as e:
                print(f"   ❌ Error loading {filename}: {str(e)}")

    def process_data(self, btn):
        """Process data, create optimization plan, and generate maps"""
        with self.status_output:
            clear_output(wait=True)

            if not all([self.warehouse_data is not None,
                       self.stock_data is not None,
                       self.orders_data is not None]):
                print("❌ Please upload all required data files first!")
                return

            print("⚡ Processing data with robust coordinate validation...")

            try:
                # Validate coordinates for warehouses and orders
                print("🔍 Validating coordinate data...")

                valid_warehouses, missing_warehouses = self.validate_coordinates(
                    self.warehouse_data, 'latitude', 'longitude'
                )
                valid_orders, missing_orders = self.validate_coordinates(
                    self.orders_data, 'Latitude', 'Longitude'
                )

                # Store missing data records
                missing_records = []
                if not missing_warehouses.empty:
                    missing_warehouses['Data_Type'] = 'Warehouse'
                    missing_records.append(missing_warehouses)

                if not missing_orders.empty:
                    missing_orders['Data_Type'] = 'Order'
                    missing_records.append(missing_orders)

                if missing_records:
                    self.missing_data_records = pd.concat(missing_records, ignore_index=True)
                    print(f"⚠️ Found {len(self.missing_data_records)} records with missing/invalid coordinates")
                else:
                    self.missing_data_records = pd.DataFrame()
                    print("✅ All coordinate data is valid")

                # Create delivery optimization plan with valid data only
                if not valid_warehouses.empty and not valid_orders.empty:
                    self.delivery_plan = self.optimize_delivery_routes(valid_warehouses, valid_orders)

                    if self.delivery_plan is not None:
                        # Create clusters and map
                        self.cluster_data, _ = self.create_clusters_and_map(valid_warehouses, valid_orders)

                        # Create route optimization
                        self.route_optimization = self.optimize_routes_by_warehouse(
                            self.delivery_plan['delivery_assignments']
                        )

                        print("✅ Processing completed successfully!")
                        print("📈 Delivery plan generated with mapping and clustering")
                        print(f"📊 Valid orders processed: {len(valid_orders)}")
                        print(f"🏢 Valid warehouses: {len(valid_warehouses)}")

                        if self.missing_data_records is not None and not self.missing_data_records.empty:
                            print(f"⚠️ Records with coordinate issues: {len(self.missing_data_records)}")

                        print("🗺️ Interactive map created: delivery_optimization_map.html")

                        # Download the map file
                        files.download('delivery_optimization_map.html')
                    else:
                        print("❌ Could not create delivery plan with available valid data!")

                else:
                    print("❌ No valid data found for processing!")
                    if valid_warehouses.empty:
                        print("   • No valid warehouse coordinates found")
                    if valid_orders.empty:
                        print("   • No valid order coordinates found")

            except Exception as e:
                print(f"❌ Error during processing: {str(e)}")
                import traceback
                print(f"   Debug info: {traceback.format_exc()}")

    def download_route_optimization(self, btn):
        """Download optimized routes by warehouse"""
        with self.status_output:
            clear_output(wait=True)

            if self.route_optimization is None:
                print("❌ Please process the data first!")
                return

            print("🚛 Generating warehouse-wise route optimization...")

            try:
                # Create Excel file with route optimization
                with pd.ExcelWriter('optimized_delivery_routes.xlsx', engine='openpyxl') as writer:

                    # Route summary
                    route_summary = []
                    for wh_code, wh_data in self.route_optimization.items():
                        route_summary.append(wh_data['route_stats'])

                    route_summary_df = pd.DataFrame(route_summary)
                    route_summary_df.to_excel(writer, sheet_name='Route_Summary', index=False)

                    # Individual warehouse routes
                    for wh_code, wh_data in self.route_optimization.items():
                        sheet_name = f'Route_{wh_code}'[:31]  # Excel sheet name limit
                        wh_data['route_details'].to_excel(writer, sheet_name=sheet_name, index=False)

                print("✅ Route optimization file generated!")
                print("📁 File: optimized_delivery_routes.xlsx")
                print("\n📊 Route details include:")
                print("   • Delivery sequence for each warehouse")
                print("   • Cumulative distances and times")
                print("   • Route efficiency scores")
                print("   • Cost optimization")

                # Display summary
                print(f"\n🚛 Route Summary:")
                for wh_code, wh_data in self.route_optimization.items():
                    stats = wh_data['route_stats']
                    print(f"   • {wh_code}: {stats['Total_Orders']} orders, "
                          f"{stats['Total_Distance_KM']} km, "
                          f"₹{stats['Total_Cost']}")

                # Download file
                files.download('optimized_delivery_routes.xlsx')
                print("\n🎉 Route optimization downloaded successfully!")

            except Exception as e:
                print(f"❌ Error generating routes: {str(e)}")

    def download_complete_analysis(self, btn):
        """Generate and download complete analysis with all data"""
        with self.status_output:
            clear_output(wait=True)

            if self.delivery_plan is None:
                print("❌ Please process the data first!")
                return

            print("📊 Generating comprehensive analysis report...")

            try:
                # Create Excel file with multiple sheets
                with pd.ExcelWriter('complete_delivery_analysis.xlsx', engine='openpyxl') as writer:

                    # Main delivery assignments
                    self.delivery_plan['delivery_assignments'].to_excel(
                        writer, sheet_name='Delivery_Assignments', index=False
                    )

                    # Missing data records
                    if self.missing_data_records is not None and not self.missing_data_records.empty:
                        self.missing_data_records.to_excel(
                            writer, sheet_name='Missing_Data', index=False
                        )
                    else:
                        # Create empty sheet with headers if no missing data
                        empty_missing = pd.DataFrame(columns=['Missing_Data_Remark'])
                        empty_missing.to_excel(writer, sheet_name='Missing_Data', index=False)

                    # Summary statistics
                    summary_df = pd.DataFrame([self.delivery_plan['summary']])
                    summary_df.to_excel(writer, sheet_name='Summary', index=False)

                    # Warehouse utilization
                    warehouse_util = self.delivery_plan['delivery_assignments'].groupby('Assigned_Warehouse').agg({
                        'Order_ID': 'count',
                        'Distance_KM': ['sum', 'mean'],
                        'Delivery_Cost': 'sum'
                    }).round(2)
                    warehouse_util.columns = ['Orders_Count', 'Total_Distance', 'Avg_Distance', 'Total_Cost']
                    warehouse_util.to_excel(writer, sheet_name='Warehouse_Utilization')

                    # State-wise analysis
                    state_analysis = self.delivery_plan['delivery_assignments'].groupby('Customer_State').agg({
                        'Order_ID': 'count',
                        'Distance_KM': 'mean',
                        'Delivery_Cost': 'sum'
                    }).round(2)
                    state_analysis.columns = ['Orders', 'Avg_Distance', 'Total_Cost']
                    state_analysis.to_excel(writer, sheet_name='State_Analysis')

                    # Cluster analysis
                    if self.cluster_data:
                        cluster_df = pd.DataFrame({
                            'Label': self.cluster_data['labels'],
                            'Type': self.cluster_data['types'],
                            'Cluster': self.cluster_data['clusters'],
                            'Latitude': [coord[0] for coord in self.cluster_data['coordinates']],
                            'Longitude': [coord[1] for coord in self.cluster_data['coordinates']]
                        })
                        cluster_df.to_excel(writer, sheet_name='Cluster_Analysis', index=False)

                    # Route optimization summary
                    if self.route_optimization:
                        route_summary = []
                        for wh_code, wh_data in self.route_optimization.items():
                            route_summary.append(wh_data['route_stats'])

                        route_summary_df = pd.DataFrame(route_summary)
                        route_summary_df.to_excel(writer, sheet_name='Route_Summary', index=False)

                print("✅ Complete analysis report generated!")
                print("📁 File: complete_delivery_analysis.xlsx")
                print("\n📊 Report includes:")
                print("   • Detailed delivery assignments")
                print("   • Missing data records with detailed remarks")
                print("   • Summary statistics")
                print("   • Warehouse utilization analysis")
                print("   • State-wise delivery analysis")
                print("   • Cluster analysis data")
                print("   • Route optimization summary")

                # Display summary
                summary = self.delivery_plan['summary']
                print(f"\n📈 Key Metrics:")
                print(f"   • Valid Orders Processed: {summary['Total_Orders']:,}")
                print(f"   • Skipped Orders (bad coordinates): {summary.get('Skipped_Orders', 0)}")
                print(f"   • Missing Records: {summary['Missing_Records']}")
                print(f"   • Total Distance: {summary['Total_Distance_KM']:,.1f} km")
                print(f"   • Total Cost: ₹{summary['Total_Cost']:,.2f}")
                print(f"   • Average Distance: {summary['Average_Distance']:.1f} km")
                print(f"   • Warehouses Used: {summary['Warehouses_Used']}")

                if self.cluster_data:
                    print(f"   • Clusters Created: {self.cluster_data['n_clusters']}")

                # Download file
                files.download('complete_delivery_analysis.xlsx')
                print("\n🎉 Complete analysis downloaded successfully!")

            except Exception as e:
                print(f"❌ Error generating complete analysis: {str(e)}")
                import traceback
                print(f"   Debug info: {traceback.format_exc()}")

# Initialize and run the application
print("🚀 Initializing Enhanced Amazon Delivery Optimization System...")
print("✨ Features: Robust coordinate validation, interactive mapping, clustering, route optimization")
print("🔧 Enhanced Error Handling: Automatically excludes invalid coordinates (NaN, 0,0, out-of-range)")
optimizer = AmazonDeliveryOptimizer()
print("✅ System ready! Use the interface above to get started.")